# 189. Native Sparse Attention：压缩块、滑窗与选择块怎样组成可训练稀疏注意力？

> **面试问题：怎样让 sparse mask 保持 causal、block 对齐且有可比的算量模型，何时必须回退 dense attention？**

## 先给结论

这题的关键不是调用框架，而是把状态、预算、验证器和可复放的制品合同显式化。教学代码用小数据验证不变量；生产实现仍需替换为真实模型、内核、隔离环境与线上观测。

## 一手资料

- [Native Sparse Attention](https://arxiv.org/abs/2502.11089)
- [Longformer](https://arxiv.org/abs/2004.05150)
- [FlashAttention](https://arxiv.org/abs/2205.14135)


In [ ]:
import hashlib  # 导入本单元依赖。
import json  # 导入本单元依赖。
import math  # 导入本单元依赖。
from dataclasses import asdict, dataclass  # 导入本单元依赖。
BLOCK = 2  # 计算并保存当前中间结果。
WINDOW = 2  # 计算并保存当前中间结果。
assert BLOCK == 2  # 用断言验证关键不变量。
assert WINDOW >= BLOCK  # 用断言验证关键不变量。
assert WINDOW % BLOCK == 0  # 用断言验证关键不变量。


## 1. 最小状态与输入合同

先说明输入输出、边界和验证 oracle；再运行下面的底层实现。


In [ ]:
def block_id(position, block):  # 定义可复用的核心函数。
    if position < 0 or block <= 0:  # 按条件选择控制路径。
        raise ValueError("位置或 block 非法")  # 非法输入立即显式失败。
    return position // block  # 返回当前计算结果。
assert block_id(0, 2) == 0  # 用断言验证关键不变量。
assert block_id(3, 2) == 1  # 用断言验证关键不变量。
assert block_id(4, 2) == 2  # 用断言验证关键不变量。


## 2. 核心公式或状态转换

先说明输入输出、边界和验证 oracle；再运行下面的底层实现。


In [ ]:
def selected_blocks(length, query, block, window, global_blocks):  # 定义可复用的核心函数。
    causal = set(range(block_id(max(0, query - window + 1), block), block_id(query, block) + 1))  # 计算并保存当前中间结果。
    return sorted(causal | set(global_blocks))  # 返回当前计算结果。
blocks = selected_blocks(8, 5, 2, 2, {0})  # 计算并保存当前中间结果。
assert blocks == [0, 2]  # 用断言验证关键不变量。
assert all(item <= block_id(5, 2) for item in blocks)  # 用断言验证关键不变量。
assert 0 in blocks  # 用断言验证关键不变量。


## 3. 候选选择与验证

先说明输入输出、边界和验证 oracle；再运行下面的底层实现。


In [ ]:
def sparse_keys(length, query, block, window, global_blocks):  # 定义可复用的核心函数。
    ids = []  # 计算并保存当前中间结果。
    for candidate in range(query + 1):  # 遍历元素以累积状态。
        if block_id(candidate, block) in selected_blocks(length, query, block, window, global_blocks):  # 按条件选择控制路径。
            ids.append(candidate)  # 计算并保存当前中间结果。
    return ids  # 返回当前计算结果。
keys = sparse_keys(8, 5, 2, 2, {0})  # 计算并保存当前中间结果。
assert keys == [0, 1, 4, 5]  # 用断言验证关键不变量。
assert all(key <= 5 for key in keys)  # 用断言验证关键不变量。
assert 2 not in keys  # 用断言验证关键不变量。


## 4. 主路径实现

先说明输入输出、边界和验证 oracle；再运行下面的底层实现。


In [ ]:
def sparse_attention(values, keys):  # 定义可复用的核心函数。
    weight = 1 / max(len(keys), 1)  # 计算并保存当前中间结果。
    return sum(values[index] * weight for index in keys)  # 返回当前计算结果。
values = list(range(8))  # 计算并保存当前中间结果。
assert math.isclose(sparse_attention(values, keys), 2.5)  # 用断言验证关键不变量。
assert sparse_attention(values, []) == 0  # 用断言验证关键不变量。
assert sparse_attention([1], [0]) == 1  # 用断言验证关键不变量。


## 5. 边界与失败分支

先说明输入输出、边界和验证 oracle；再运行下面的底层实现。


In [ ]:
dense_keys = sparse_keys(6, 5, 2, 6, {0, 1, 2})  # 计算并保存当前中间结果。
assert dense_keys == list(range(6))  # 用断言验证关键不变量。
assert sparse_attention(values[:6], dense_keys) == sum(values[:6]) / 6  # 用断言验证关键不变量。
assert len(dense_keys) == 6  # 用断言验证关键不变量。


## 6. 正确性与基线对照

先说明输入输出、边界和验证 oracle；再运行下面的底层实现。


In [ ]:
def attention_work(length, selected_key_count):  # 定义可复用的核心函数。
    return {"dense": length * length, "sparse": length * selected_key_count}  # 返回当前计算结果。
work = attention_work(8, len(keys))  # 计算并保存当前中间结果。
assert work["sparse"] < work["dense"]  # 用断言验证关键不变量。
assert work["dense"] == 64  # 用断言验证关键不变量。
assert work["sparse"] == 32  # 用断言验证关键不变量。


## 7. 成本或数据合同

先说明输入输出、边界和验证 oracle；再运行下面的底层实现。


In [ ]:
@dataclass(frozen=True)  # 计算并保存当前中间结果。
class NSAArtifact:  # 定义保存状态的数据结构。
    block: int  # 计算并保存当前中间结果。
    window: int  # 计算并保存当前中间结果。
    global_policy: str  # 计算并保存当前中间结果。
artifact = NSAArtifact(BLOCK, WINDOW, "compressed-block-score")  # 计算并保存当前中间结果。
assert artifact.block == 2  # 用断言验证关键不变量。
assert artifact.window == 2  # 用断言验证关键不变量。
assert "block" in artifact.global_policy  # 用断言验证关键不变量。


## 8. 制品版本与面试收束

先说明输入输出、边界和验证 oracle；再运行下面的底层实现。


In [ ]:
assert selected_blocks(4, 0, 2, 2, set()) == [0]  # 用断言验证关键不变量。
assert sparse_keys(4, 0, 2, 2, set()) == [0]  # 用断言验证关键不变量。
assert block_id(7, 2) == 3  # 用断言验证关键不变量。


## 面试收束

回答时按目标、状态合同、核心算法、失败分支、评测指标与发布版本组织；受控例子只证明实现不变量，不代表真实模型或生产系统性能。
